# Game Design Mentor Agent

## Step 1: 에이전트 설계

- 이름: `GameDesignMentorAgent`
- 목적: 초보 인디 개발자가 작성한 게임 기획 초안을 읽고, 무엇이 빠졌는지와 무엇을 먼저 정해야 하는지를 한 번의 리뷰로 알려줍니다.
- 핵심 기능:
  1. 자유형 기획 메모에서 핵심 필드(콘셉트, 대상 플레이어, 감정 목표, 코어 루프, 기능 목록)를 추출합니다.
  2. `감정 목표`, `코어 루프` 같은 본 리뷰 필수 항목 누락 여부를 판단하고, 부족하면 질문 중심 응답으로 전환합니다.
  3. 리뷰 가능 상태면 의도 정렬, 코어 루프, MVP 범위, 플레이테스트 질문을 묶어 짧은 진단을 생성합니다.
  4. 정확히 2개의 해석 방향과 각 방향의 트레이드오프를 제시합니다.

## MVP 그래프 구조

실제 MVP 설계는 아래 흐름을 기준으로 잡습니다.

```text
[START]
   |
   v
[Concept Intake / Extract]
   |
   v
[Validate Required Fields]
   |
   +--> (missing core_loop or emotion_goal)
   |         |
   |         v
   |     [Clarify Missing]
   |
   +--> (review ready)
             |
             v
      [Intent Alignment Review]
             |
             v
      [Core Loop Review]
             |
             v
      [MVP Scope / Playtest Review]
             |
             v
      [Direction Compare + Final Output]
             |
             v
            [END]
```

### 노드 책임

- `Concept Intake / Extract`: 자유형 메모를 리뷰 가능한 필드로 구조화
- `Validate Required Fields`: 필수값 누락 여부 판단
- `Clarify Missing`: 본 리뷰 대신 질문 반환
- `Intent Alignment Review`: 대상 플레이어, 감정 목표, 콘셉트 정렬 점검
- `Core Loop Review`: 반복 구조, 보상 구조, 차별화 검토
- `MVP Scope / Playtest Review`: 지금 뺄 것, 검증 가설, 테스트 질문 정리
- `Direction Compare + Final Output`: 2개 방향, 트레이드오프, 다음 결정 1개로 마무리


In [11]:
import os
from typing import TypedDict

from langchain.chat_models import init_chat_model
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()

True

In [12]:
class MentorState(TypedDict, total=False):
    raw_input: str
    concept_statement: str
    target_player: str
    emotion_goal: str
    core_loop: str
    feature_list: list[str]
    missing_fields: list[str]
    review_ready: bool


class ExtractedBrief(BaseModel):
    concept_statement: str = Field(
        default="", description="게임 콘셉트를 한 줄로 요약한 문장"
    )
    target_player: str = Field(
        default="", description="가장 우선 타깃으로 보는 플레이어"
    )
    emotion_goal: str = Field(
        default="", description="플레이어에게 주고 싶은 핵심 감정"
    )
    core_loop: str = Field(default="", description="반복 플레이의 핵심 루프")
    feature_list: list[str] = Field(
        default_factory=list, description="명시된 주요 기능 목록"
    )

In [13]:
llm = init_chat_model("openai:gpt-4o-mini", temperature=0).with_structured_output(
    ExtractedBrief
)


def extract_brief(state: MentorState) -> MentorState:
    prompt = f"""
You extract a game design brief from free-form Korean markdown notes.

Rules:
- Return only the structured fields requested by the schema.
- If a field is missing or unclear, return an empty string.
- Keep `feature_list` to explicit features only.
- Do not invent game details.

Draft:
{state['raw_input']}
""".strip()

    extracted = llm.invoke(prompt)
    return extracted.model_dump()


def validate_required_fields(state: MentorState) -> MentorState:
    missing_fields = [
        name for name in ["emotion_goal", "core_loop"] if not state.get(name)
    ]
    return {
        "missing_fields": missing_fields,
        "review_ready": not missing_fields,
    }

In [14]:
graph_builder = StateGraph(MentorState)
graph_builder.add_node("extract_brief", extract_brief)
graph_builder.add_node("validate_required_fields", validate_required_fields)

graph_builder.add_edge(START, "extract_brief")
graph_builder.add_edge("extract_brief", "validate_required_fields")
graph_builder.add_edge("validate_required_fields", END)

graph = graph_builder.compile()

In [15]:
sample_input = {
    "raw_input": """이 게임은 전투보다 생존 판단이 중요한 설산 탐험 게임이다.
짧은 세션의 전략 게임을 좋아하는 직장인을 먼저 노린다.
플레이어에게는 불안하지만 한 턴만 더 해보고 싶은 긴장감을 주고 싶다.
핵심 루프는 이동한다 -> 자원을 고른다 -> 날씨 리스크를 버틴다 -> 다음 거점까지 간다.
기능 후보:
- 튜토리얼
- 거점 업그레이드
- 날씨 이벤트
"""
}

result = graph.invoke(sample_input)
print(result)

{'raw_input': '이 게임은 전투보다 생존 판단이 중요한 설산 탐험 게임이다.\n짧은 세션의 전략 게임을 좋아하는 직장인을 먼저 노린다.\n플레이어에게는 불안하지만 한 턴만 더 해보고 싶은 긴장감을 주고 싶다.\n핵심 루프는 이동한다 -> 자원을 고른다 -> 날씨 리스크를 버틴다 -> 다음 거점까지 간다.\n기능 후보:\n- 튜토리얼\n- 거점 업그레이드\n- 날씨 이벤트\n', 'concept_statement': '이 게임은 전투보다 생존 판단이 중요한 설산 탐험 게임이다.', 'target_player': '짧은 세션의 전략 게임을 좋아하는 직장인', 'emotion_goal': '플레이어에게는 불안하지만 한 턴만 더 해보고 싶은 긴장감을 주고 싶다.', 'core_loop': '이동한다 -> 자원을 고른다 -> 날씨 리스크를 버틴다 -> 다음 거점까지 간다.', 'feature_list': ['튜토리얼', '거점 업그레이드', '날씨 이벤트'], 'missing_fields': [], 'review_ready': True}


/Users/hojun/Development/game-design-mentor/.venv/lib/python3.13/site-packages/pydantic/main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=ExtractedBrief(concept_st...', '날씨 이벤트']), input_type=ExtractedBrief])
  return self.__pydantic_serializer__.to_python(
